[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/27.%20The%20Integrated%20Berth%20%26%20Crane%20Allocation%20Problem%20%28BAP-QCAP%29/P27-Tier-2_executed.ipynb)

---


# 27. The Integrated Berth and Crane Allocation Problem (BAP-QCAP)

## Tier 2: Priority-Based Constructive Heuristic

### Goal and scope

This notebook implements a priority-based constructive heuristic for an integrated berth and quay-crane allocation problem. It processes vessels in descending priority order, selects a berth position from a discrete set, chooses a crane count using the stated efficiency rule, and records each vessel's berth occupancy on a timeline. The result is a feasible schedule under the assumptions below; the heuristic does not guarantee a globally optimal solution.

The inputs are vessel lengths, workloads, arrival times, hourly costs and crane limits, together with quay length and crane-productivity parameters. The outputs are vessel assignments, cost and time summaries, visualizations, and a reproducible scalability experiment. The planning horizon is recorded as instance data but is not enforced as a hard deadline.

### Model assumptions

- Time is measured in hours from the start of a common planning day. Arrival times may be followed by service that finishes after that day.
- The quay is a one-dimensional interval. A vessel occupies the interval from its assigned position to that position plus its length throughout service.
- Candidate berth positions are multiples of the specified position step. A position is feasible only if the vessel fits within the quay.
- A vessel may not overlap another vessel in both space and service time. Berth occupancy ends when service finishes.
- Each vessel's service productivity is the sum of its assigned cranes' productivities after the modeled interference reductions. Crane interference is applied by crane index within each vessel's assignment; crane availability across vessels is not constrained.
- Priority is workload divided by vessel length and the remaining time until hour 24, with a minimum time-window denominator of one hour. Equal priorities retain the vessels' input order.
- The assignment cost is hourly vessel cost multiplied by waiting time plus service time. There are no separate crane, berth, or lateness costs.

### Method overview

The heuristic first sorts vessels by priority. For each vessel, it evaluates candidate berth positions, finds the earliest non-conflicting start at each position, and selects the position with the lowest resulting vessel cost. It then records the assignment so later vessels can account for its berth occupancy. The crane count is selected by maximizing average productivity per crane, subject to the vessel's crane limit and the implementation's practical cap.

The position search is discrete, so the heuristic considers only the specified grid of berth positions. Its result can depend on the priority rule, position step, and greedy processing order. It is a constructive heuristic, not an exact optimization procedure, and no fixed percentage gap from an optimum is asserted.

### Priority rule

For vessel $j$, the priority used for sequencing is

$$\pi_j = \frac{w_j}{\ell_j \max(1, 24-a_j)}.$$

Here, $w_j$ is workload in TEU, $\ell_j$ is vessel length in meters, and $a_j$ is arrival time in hours from the start of the planning day. The denominator combines vessel length with a time window that decreases as arrival approaches hour 24. A larger value places the vessel earlier in the sequence. In the code, `Vessel.__post_init__` computes this value and `PriorityBasedHeuristic.solve` sorts by it.

### Crane productivity and service time

With $k$ cranes, the modeled total productivity is

$$P(k) = p\sum_{i=0}^{k-1}(1-\alpha i),$$

and the service time for vessel $j$ is

$$s_j(k) = \frac{w_j}{P(k)}.$$

Here, $p$ is the productivity of one crane in TEU per hour and $\alpha$ is the interference factor. The implementation chooses the crane count that maximizes $P(k)/k$, the average productivity per crane, among the allowed counts. It then uses `calculate_service_time` to calculate duration. This efficiency rule is not the same as minimizing service time or total cost; the assignment cost has no crane-count penalty.

### Assignment cost and berth-time feasibility

For a candidate start time $t_j$, arrival time $a_j$, and service duration $s_j$, waiting time and turnaround time are

$$q_j = t_j-a_j, \qquad T_j=q_j+s_j,$$

and the assignment cost is

$$C_j=c_jT_j,$$

where $c_j$ is the vessel's cost per hour. For a candidate berth interval $[x_j,x_j+\ell_j)$, a start is feasible only if its service interval does not overlap the service interval of any vessel occupying an overlapping berth interval. `BerthTimeline.earliest_available` searches for the earliest such start.

The heuristic selects the feasible candidate with the smallest vessel cost. Since service duration and hourly cost are fixed across the candidate positions for a vessel, this is equivalent to selecting the candidate with the least waiting time. Ties retain the first candidate position in the search order.

### Computational effort

Let $n$ be the number of vessels, $P$ the number of candidate berth positions, and $K$ the maximum number of crane counts tested per vessel. Sorting costs $O(n\log n)$. For each vessel, the position search checks up to $P$ positions against up to $n$ existing occupancies, and the crane search checks up to $K$ counts. The resulting upper-bound work is $O(n\log n+n^2P+nK)$, excluding the small cost of sorting the occupancy list after each insertion. With a fixed quay and fixed position step, $P$ is bounded independently of $n$, giving quadratic growth in the vessel count as the dominant term. This describes the implemented search, not a guarantee about runtime on every machine.

### How to read the results

The schedule reports times as decimal hours from the planning-day origin, not as formatted clock times. The reported aggregate berth-length ratio is the sum of assigned vessel lengths divided by quay length; it is not a time-based utilization measure and can exceed 100% when vessels use the quay at different times. Runtime measurements are machine- and environment-dependent. The scalability experiment uses fixed random seeds so its generated instances are reproducible.

In [ ]:
# Import the libraries used for the heuristic, data structures, timing, and plots.
import math
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from matplotlib.patches import Patch, Rectangle

plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("Libraries imported successfully.")

In [ ]:
@dataclass
class Vessel:
    """Data for a vessel that requires berth and crane service."""

    id: int
    length: float  # meters
    workload: float  # TEU
    arrival_time: float  # hours from the planning-day origin
    cost_per_hour: float = 1000.0
    max_cranes: int = 5

    def __post_init__(self) -> None:
        if self.length <= 0 or self.workload <= 0:
            raise ValueError("Vessel length and workload must be positive.")
        if self.arrival_time < 0 or self.cost_per_hour < 0:
            raise ValueError("Arrival time and hourly cost must be non-negative.")
        if self.max_cranes < 1:
            raise ValueError("Each vessel must allow at least one crane.")

        time_window = max(1.0, 24.0 - self.arrival_time)
        self.priority = self.workload / (self.length * time_window)


@dataclass
class BerthAssignment:
    """A vessel's position and service interval on the quay."""

    vessel: Vessel
    position: float
    start_time: float
    finish_time: float
    num_cranes: int
    service_time: float
    waiting_time: float
    total_time: float

    @property
    def cost(self) -> float:
        """Return the vessel's hourly cost multiplied by turnaround time."""
        return self.total_time * self.vessel.cost_per_hour


@dataclass
class BerthTimeline:
    """Store berth intervals and find non-conflicting service starts."""

    quay_length: float

    def __post_init__(self) -> None:
        self.occupancy: List[Tuple[float, float, float, float]] = []

    def earliest_available(
        self,
        position: float,
        length: float,
        arrival_time: float,
        service_time: float,
    ) -> float:
        """Find the earliest start whose full service interval is feasible."""
        candidate_start = arrival_time

        for occ_start, occ_end, occ_position, occ_length in self.occupancy:
            spatially_overlaps = not (
                position + length <= occ_position
                or occ_position + occ_length <= position
            )
            temporally_overlaps = (
                candidate_start < occ_end
                and candidate_start + service_time > occ_start
            )

            if spatially_overlaps and temporally_overlaps:
                candidate_start = occ_end

        return candidate_start

    def add_occupancy(
        self,
        position: float,
        length: float,
        start_time: float,
        end_time: float,
    ) -> None:
        """Record a completed assignment for later conflict checks."""
        self.occupancy.append((start_time, end_time, position, length))
        self.occupancy.sort(key=lambda item: item[0])


@dataclass
class BAPQCAPInstance:
    """Problem data for the integrated berth and crane allocation instance."""

    vessels: List[Vessel]
    quay_length: float
    interference_factor: float
    crane_productivity: float
    planning_horizon: float

    def __post_init__(self) -> None:
        if self.quay_length <= 0 or self.planning_horizon <= 0:
            raise ValueError("Quay length and planning horizon must be positive.")
        if self.crane_productivity <= 0:
            raise ValueError("Crane productivity must be positive.")
        if not 0 <= self.interference_factor < 1:
            raise ValueError("The interference factor must be in [0, 1).")
        if any(vessel.length > self.quay_length for vessel in self.vessels):
            raise ValueError("Every vessel must fit within the quay length.")


print("Problem data structures are ready.")

In [ ]:
def create_concrete_example() -> BAPQCAPInstance:
    """Create the three-vessel instance used to demonstrate the heuristic."""
    vessels = [
        Vessel(id=1, length=300, workload=800, arrival_time=8.0),
        Vessel(id=2, length=300, workload=1200, arrival_time=10.0),
        Vessel(id=3, length=200, workload=400, arrival_time=12.0),
    ]

    return BAPQCAPInstance(
        vessels=vessels,
        quay_length=1200,
        interference_factor=0.1,
        crane_productivity=30,
        planning_horizon=48,
    )


instance = create_concrete_example()

print("=== Problem Instance ===")
print(f"Quay length: {instance.quay_length:.0f} m")
print(f"Crane productivity: {instance.crane_productivity:.1f} TEU/hour per crane")
print(f"Interference factor: {instance.interference_factor:.2f}")
print(f"Planning horizon (not a hard deadline): {instance.planning_horizon:.1f} hours")
print("\nVessel data:")
for vessel in instance.vessels:
    print(
        f"Vessel {vessel.id}: length={vessel.length:.0f} m, "
        f"workload={vessel.workload:.0f} TEU, "
        f"arrival={vessel.arrival_time:.1f} h, "
        f"priority={vessel.priority:.4f}, "
        f"maximum cranes={vessel.max_cranes}"
    )

In [ ]:
class PriorityBasedHeuristic:
    """Construct a berth and crane schedule using the priority rule."""

    def __init__(self, instance: BAPQCAPInstance):
        self.instance = instance
        self.timeline = BerthTimeline(instance.quay_length)
        self.assignments: List[BerthAssignment] = []

    def calculate_service_time(self, vessel: Vessel, num_cranes: int) -> float:
        """Calculate duration from workload and interference-adjusted output."""
        if num_cranes < 1:
            return float("inf")

        total_productivity = sum(
            self.instance.crane_productivity
            * (1 - self.instance.interference_factor * crane_index)
            for crane_index in range(num_cranes)
        )
        if total_productivity <= 0:
            return float("inf")

        return vessel.workload / total_productivity

    def optimize_crane_count(self, vessel: Vessel) -> int:
        """Choose the allowed count with greatest average productivity per crane."""
        best_count = 1
        best_efficiency = float("-inf")
        max_possible = min(vessel.max_cranes, 8)

        for count in range(1, max_possible + 1):
            total_productivity = sum(
                self.instance.crane_productivity
                * (1 - self.instance.interference_factor * crane_index)
                for crane_index in range(count)
            )
            efficiency = total_productivity / count

            if efficiency > best_efficiency:
                best_efficiency = efficiency
                best_count = count

        return best_count

    def find_best_position(self, vessel: Vessel) -> Tuple[float, float]:
        """Choose the candidate position with the lowest waiting-related cost."""
        position_step = 10
        max_position = self.instance.quay_length - vessel.length
        last_grid_position = int(max_position // position_step) * position_step
        num_cranes = self.optimize_crane_count(vessel)
        service_time = self.calculate_service_time(vessel, num_cranes)

        best_position = 0.0
        best_start_time = float("inf")
        best_cost = float("inf")

        for position in range(0, last_grid_position + 1, position_step):
            start_time = self.timeline.earliest_available(
                position=position,
                length=vessel.length,
                arrival_time=vessel.arrival_time,
                service_time=service_time,
            )
            waiting_time = start_time - vessel.arrival_time
            candidate_cost = (waiting_time + service_time) * vessel.cost_per_hour

            if candidate_cost < best_cost:
                best_cost = candidate_cost
                best_position = float(position)
                best_start_time = start_time

        return best_position, best_start_time

    def solve(self) -> List[BerthAssignment]:
        """Sequence vessels by priority and add feasible assignments in turn."""
        print("=== Priority-Based Constructive Heuristic ===")
        sorted_vessels = sorted(
            self.instance.vessels,
            key=lambda vessel: vessel.priority,
            reverse=True,
        )

        print("\nVessel processing sequence (descending priority):")
        for rank, vessel in enumerate(sorted_vessels, start=1):
            print(f"  {rank}. Vessel {vessel.id} (priority: {vessel.priority:.4f})")

        for vessel in sorted_vessels:
            position, start_time = self.find_best_position(vessel)
            num_cranes = self.optimize_crane_count(vessel)
            service_time = self.calculate_service_time(vessel, num_cranes)
            finish_time = start_time + service_time
            waiting_time = start_time - vessel.arrival_time
            total_time = waiting_time + service_time

            assignment = BerthAssignment(
                vessel=vessel,
                position=position,
                start_time=start_time,
                finish_time=finish_time,
                num_cranes=num_cranes,
                service_time=service_time,
                waiting_time=waiting_time,
                total_time=total_time,
            )

            self.timeline.add_occupancy(
                position,
                vessel.length,
                start_time,
                finish_time,
            )
            self.assignments.append(assignment)

            print(f"\nVessel {vessel.id}:")
            print(
                f"  Berth interval: [{position:.0f}, "
                f"{position + vessel.length:.0f}] m"
            )
            print(
                f"  Start: {start_time:.2f} h; "
                f"finish: {finish_time:.2f} h"
            )
            print(
                f"  Cranes: {num_cranes}; "
                f"service: {service_time:.2f} h; "
                f"waiting: {waiting_time:.2f} h"
            )
            print(f"  Turnaround: {total_time:.2f} h; cost: ${assignment.cost:,.2f}")

        self.validate_assignments()
        return self.assignments

    def validate_assignments(self) -> None:
        """Check assignment coverage, bounds, time consistency, and conflicts."""
        expected_ids = {vessel.id for vessel in self.instance.vessels}
        assigned_ids = [assignment.vessel.id for assignment in self.assignments]
        assert set(assigned_ids) == expected_ids
        assert len(assigned_ids) == len(expected_ids)

        for assignment in self.assignments:
            vessel = assignment.vessel
            assert assignment.position >= 0
            assert assignment.position + vessel.length <= self.instance.quay_length
            assert assignment.start_time >= vessel.arrival_time
            assert math.isclose(
                assignment.finish_time,
                assignment.start_time + assignment.service_time,
                rel_tol=1e-9,
                abs_tol=1e-9,
            )
            assert math.isclose(
                assignment.total_time,
                assignment.waiting_time + assignment.service_time,
                rel_tol=1e-9,
                abs_tol=1e-9,
            )

        for index, first in enumerate(self.assignments):
            for second in self.assignments[index + 1:]:
                spatially_overlaps = not (
                    first.position + first.vessel.length <= second.position
                    or second.position + second.vessel.length <= first.position
                )
                temporally_overlaps = not (
                    first.finish_time <= second.start_time
                    or second.finish_time <= first.start_time
                )
                assert not (spatially_overlaps and temporally_overlaps)

    def get_performance_metrics(self) -> Dict[str, float]:
        """Summarize cost, time, crane count, and aggregate length ratio."""
        if not self.assignments:
            return {}

        num_vessels = len(self.assignments)
        total_cost = sum(assignment.cost for assignment in self.assignments)
        total_waiting = sum(
            assignment.waiting_time for assignment in self.assignments
        )
        total_service = sum(
            assignment.service_time for assignment in self.assignments
        )
        total_time = sum(assignment.total_time for assignment in self.assignments)
        total_cranes = sum(
            assignment.num_cranes for assignment in self.assignments
        )
        aggregate_length_ratio = (
            sum(assignment.vessel.length for assignment in self.assignments)
            / self.instance.quay_length
        )

        return {
            "total_cost": total_cost,
            "total_waiting_time": total_waiting,
            "total_service_time": total_service,
            "total_turnaround_time": total_time,
            "total_cranes_used": total_cranes,
            "aggregate_berth_length_ratio": aggregate_length_ratio,
            "avg_waiting_time": total_waiting / num_vessels,
            "avg_service_time": total_service / num_vessels,
            "avg_turnaround_time": total_time / num_vessels,
            "num_vessels": num_vessels,
        }


print("Priority-based heuristic and validation checks are ready.")

In [ ]:
def solve_with_heuristic(instance: BAPQCAPInstance):
    """Run the priority heuristic and report its measured runtime and metrics."""
    start_time = time.perf_counter()
    heuristic = PriorityBasedHeuristic(instance)
    assignments = heuristic.solve()
    computation_time = time.perf_counter() - start_time
    metrics = heuristic.get_performance_metrics()

    print("\n=== Heuristic Solution Summary ===")
    print(f"Measured computation time: {computation_time:.6f} seconds")
    print(f"Total cost: ${metrics['total_cost']:,.2f}")
    print(f"Total turnaround time: {metrics['total_turnaround_time']:.2f} hours")
    print(f"Average turnaround time: {metrics['avg_turnaround_time']:.2f} hours")
    print(
        "Aggregate berth-length ratio: "
        f"{metrics['aggregate_berth_length_ratio'] * 100:.2f}%"
    )
    print(f"Sum of vessel crane assignments: {metrics['total_cranes_used']}")

    return heuristic, assignments, metrics, computation_time


heuristic, assignments, metrics, comp_time = solve_with_heuristic(instance)

In [ ]:
def visualize_heuristic_solution(
    instance: BAPQCAPInstance,
    assignments: List[BerthAssignment],
    metrics: Dict[str, float],
    computation_time: float,
) -> None:
    """Plot berth positions, service times, priorities, and solution metrics."""
    if not assignments:
        print("There are no assignments to visualize.")
        return

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    fig.suptitle("Priority-Based Heuristic Results", fontsize=16, fontweight="bold")
    colors = plt.cm.Set3(np.linspace(0, 1, len(assignments)))

    # This view shows berth positions only; it does not encode service time.
    ax1 = axes[0, 0]
    ax1.set_title("Berth Position Assignments (time not shown)", fontweight="bold")
    ax1.set_xlabel("Berth position (m)")
    ax1.set_ylabel("Vessel ID")
    for index, assignment in enumerate(assignments):
        vessel = assignment.vessel
        rectangle = Rectangle(
            (assignment.position, vessel.id - 0.4),
            vessel.length,
            0.8,
            facecolor=colors[index],
            edgecolor="black",
            linewidth=1.5,
        )
        ax1.add_patch(rectangle)
        ax1.text(
            assignment.position + vessel.length / 2,
            vessel.id,
            f"Vessel {vessel.id}\n{assignment.num_cranes} cranes",
            ha="center",
            va="center",
            fontsize=9,
        )
    ax1.set_xlim(0, instance.quay_length)
    ax1.set_ylim(0, max(assignment.vessel.id for assignment in assignments) + 1)
    ax1.set_yticks([assignment.vessel.id for assignment in assignments])
    ax1.grid(True, alpha=0.3)

    # Service bars and waiting bars use the same hour-based time axis.
    ax2 = axes[0, 1]
    ax2.set_title("Vessel Service Schedule", fontweight="bold")
    ax2.set_xlabel("Time from planning-day origin (hours)")
    ax2.set_ylabel("Vessel ID")
    for index, assignment in enumerate(assignments):
        vessel = assignment.vessel
        ax2.barh(
            vessel.id,
            assignment.service_time,
            left=assignment.start_time,
            height=0.6,
            color=colors[index],
            edgecolor="black",
        )
        if assignment.waiting_time > 0:
            ax2.barh(
                vessel.id,
                assignment.waiting_time,
                left=vessel.arrival_time,
                height=0.3,
                color="red",
                alpha=0.5,
            )
        ax2.axvline(
            x=vessel.arrival_time,
            color="red",
            linestyle="--",
            alpha=0.3,
        )
    ax2.set_xlim(left=0)
    ax2.set_yticks([assignment.vessel.id for assignment in assignments])
    ax2.grid(True, alpha=0.3)
    ax2.legend(
        handles=[
            Patch(facecolor="gray", edgecolor="black", label="Service"),
            Patch(facecolor="red", alpha=0.5, label="Waiting"),
            plt.Line2D([0], [0], color="red", linestyle="--", label="Arrival"),
        ],
        loc="best",
    )

    ax3 = axes[1, 0]
    ax3.set_title("Priority and Turnaround Time", fontweight="bold")
    for assignment in assignments:
        ax3.scatter(
            assignment.vessel.priority,
            assignment.total_time,
            s=100,
            edgecolors="black",
        )
        ax3.annotate(
            f"Vessel {assignment.vessel.id}",
            (assignment.vessel.priority, assignment.total_time),
            xytext=(5, 5),
            textcoords="offset points",
        )
    ax3.set_xlabel("Priority")
    ax3.set_ylabel("Turnaround time (hours)")
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    ax4.set_title("Performance Metrics", fontweight="bold")
    ax4.axis("off")
    metrics_text = (
        "SOLUTION SUMMARY\n"
        "------------------------------\n"
        f"Total cost: ${metrics['total_cost']:,.2f}\n"
        f"Average turnaround: {metrics['avg_turnaround_time']:.2f} h\n"
        "Aggregate berth-length ratio: "
        f"{metrics['aggregate_berth_length_ratio'] * 100:.2f}%\n"
        f"Sum of crane assignments: {metrics['total_cranes_used']}\n\n"
        "TIME COMPONENTS\n"
        "------------------------------\n"
        f"Average waiting: {metrics['avg_waiting_time']:.2f} h\n"
        f"Average service: {metrics['avg_service_time']:.2f} h\n"
        f"Total waiting: {metrics['total_waiting_time']:.2f} h\n"
        f"Total service: {metrics['total_service_time']:.2f} h\n\n"
        "COMPUTATION\n"
        "------------------------------\n"
        f"Measured runtime: {computation_time:.6f} s\n"
        f"Vessels: {metrics['num_vessels']}"
    )
    ax4.text(
        0.05,
        0.95,
        metrics_text,
        transform=ax4.transAxes,
        fontsize=10,
        verticalalignment="top",
        fontfamily="monospace",
        bbox={"boxstyle": "round", "facecolor": "lightgray", "alpha": 0.8},
    )

    plt.tight_layout()
    plt.show()


visualize_heuristic_solution(instance, assignments, metrics, comp_time)

In [ ]:
def summarize_assignments(assignments: List[BerthAssignment]) -> None:
    """Display each assignment's time and cost components for interpretation."""
    if not assignments:
        print("There are no assignments to summarize.")
        return

    print("Vessel | Position (m) | Start (h) | Service (h) | Waiting (h) | Cost ($)")
    print("-" * 78)
    for assignment in assignments:
        print(
            f"{assignment.vessel.id:>6} | "
            f"{assignment.position:>12.0f} | "
            f"{assignment.start_time:>9.2f} | "
            f"{assignment.service_time:>11.2f} | "
            f"{assignment.waiting_time:>11.2f} | "
            f"{assignment.cost:>9,.2f}"
        )

    assignment_cost = sum(assignment.cost for assignment in assignments)
    metric_cost = metrics["total_cost"]
    assert math.isclose(assignment_cost, metric_cost, rel_tol=1e-9, abs_tol=1e-7)
    print("-" * 78)
    print(f"Sum of assignment costs: ${assignment_cost:,.2f}")
    print("Validation: assignment costs sum to the reported total cost.")


summarize_assignments(assignments)

In [ ]:
def scalability_analysis():
    """Measure the same heuristic on reproducible instances of increasing size."""
    print("=== Scalability Analysis ===")
    vessel_counts = [3, 5, 8, 10, 15, 20]
    results = []

    for num_vessels in vessel_counts:
        np.random.seed(42 + num_vessels)
        vessels = []
        for vessel_index in range(num_vessels):
            vessels.append(
                Vessel(
                    id=vessel_index + 1,
                    length=np.random.uniform(150, 400),
                    workload=np.random.uniform(200, 1500),
                    arrival_time=np.random.uniform(0, 24),
                    cost_per_hour=1000,
                )
            )

        test_instance = BAPQCAPInstance(
            vessels=vessels,
            quay_length=1200,
            interference_factor=0.1,
            crane_productivity=30,
            planning_horizon=72,
        )

        start_time = time.perf_counter()
        test_heuristic = PriorityBasedHeuristic(test_instance)
        test_assignments = test_heuristic.solve()
        computation_time = time.perf_counter() - start_time
        test_metrics = test_heuristic.get_performance_metrics()

        results.append(
            {
                "vessels": num_vessels,
                "computation_time": computation_time,
                "total_cost": test_metrics["total_cost"],
                "average_turnaround": test_metrics["avg_turnaround_time"],
                "aggregate_berth_length_ratio": test_metrics[
                    "aggregate_berth_length_ratio"
                ],
            }
        )
        print(
            f"{num_vessels} vessels: {computation_time:.6f} s, "
            f"${test_metrics['total_cost']:,.2f} total cost, "
            f"{test_metrics['avg_turnaround_time']:.2f} h average turnaround"
        )

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.suptitle("Heuristic Scalability Analysis", fontsize=16, fontweight="bold")
    counts = [result["vessels"] for result in results]

    ax1 = axes[0, 0]
    runtimes_ms = [result["computation_time"] * 1000 for result in results]
    ax1.plot(counts, runtimes_ms, "bo-", linewidth=2, markersize=7)
    ax1.set_xlabel("Number of vessels")
    ax1.set_ylabel("Measured computation time (ms)")
    ax1.set_title("Runtime by Instance Size")
    ax1.grid(True, alpha=0.3)
    if all(runtime > 0 for runtime in runtimes_ms):
        ax1.set_yscale("log")

    ax2 = axes[0, 1]
    average_turnaround = [result["average_turnaround"] for result in results]
    ax2.plot(counts, average_turnaround, "ro-", linewidth=2, markersize=7)
    ax2.set_xlabel("Number of vessels")
    ax2.set_ylabel("Average turnaround time (hours)")
    ax2.set_title("Average Turnaround by Instance Size")
    ax2.grid(True, alpha=0.3)

    ax3 = axes[1, 0]
    length_ratios = [
        result["aggregate_berth_length_ratio"] * 100 for result in results
    ]
    ax3.plot(counts, length_ratios, "go-", linewidth=2, markersize=7)
    ax3.set_xlabel("Number of vessels")
    ax3.set_ylabel("Aggregate vessel-length / quay-length (%)")
    ax3.set_title("Aggregate Berth-Length Ratio")
    ax3.grid(True, alpha=0.3)

    ax4 = axes[1, 1]
    theoretical = [count ** 2 for count in counts]
    actual = [result["computation_time"] for result in results]
    max_theoretical = max(theoretical)
    max_actual = max(actual)
    theoretical_normalized = [value / max_theoretical for value in theoretical]
    actual_normalized = [value / max_actual for value in actual]
    ax4.plot(
        counts,
        theoretical_normalized,
        "b--",
        label="Quadratic reference (fixed position grid)",
        linewidth=2,
    )
    ax4.plot(
        counts,
        actual_normalized,
        "r-o",
        label="Measured runtime (normalized)",
        linewidth=2,
        markersize=5,
    )
    ax4.set_xlabel("Number of vessels")
    ax4.set_ylabel("Normalized value")
    ax4.set_title("Quadratic Reference and Measured Runtime")
    ax4.legend()
    ax4.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    print("\nScalability summary:")
    print("Vessels | Runtime (ms) | Total cost ($) | Avg. turnaround (h)")
    print("-" * 68)
    for result in results:
        print(
            f"{result['vessels']:>7} | "
            f"{result['computation_time'] * 1000:>12.4f} | "
            f"{result['total_cost']:>14,.2f} | "
            f"{result['average_turnaround']:>19.2f}"
        )

    return results


scalability_results = scalability_analysis()

## Interpreting the heuristic

### Strengths and limitations

The method is constructive: once a vessel has been assigned, its berth interval and service interval are added to the timeline, and later assignments respect that occupancy. This makes the decision sequence explicit and produces a schedule without enumerating complete schedules. The search is limited to the priority order and the candidate-position grid, so the result is not guaranteed to be optimal. Early assignments are not reconsidered.

The priority rule is a modeling choice rather than a universal measure of urgency. It combines workload, vessel length, and time remaining in the planning day; it does not directly optimize the sum of assignment costs. Likewise, the crane rule maximizes average productivity per crane, while the assignment cost contains no crane-related charge. These distinctions matter when interpreting the output: a high priority does not prove that a vessel receives the lowest cost, and an efficiency-selected crane count does not prove that total turnaround cost is minimized.

The model treats crane productivity independently for each vessel and does not limit the number of cranes available to the terminal as a whole. It also omits tidal restrictions, safety clearances, draft compatibility, setup times, vessel-specific berth preferences, and uncertainty in arrivals or handling rates. Those features can be incorporated as additional modeling requirements, but they are outside this notebook's scope.

### When this heuristic is informative

This implementation is useful for tracing how a priority rule, a discrete berth-position search, and a berth-occupancy timeline combine to construct a schedule. The scalability experiment measures runtime and schedule summaries for generated instances; it does not establish optimality or a percentage gap from an exact solution. Use the displayed assumptions and limitations when deciding whether its assignments represent an operational setting.

In [ ]:
def final_heuristic_summary(
    instance: BAPQCAPInstance,
    assignments: List[BerthAssignment],
    metrics: Dict[str, float],
    computation_time: float,
) -> None:
    """Report the solution, its construction rule, and its modeling limits."""
    print("=== Priority-Based Heuristic Summary ===")
    print("\nMethod:")
    print("  Vessel sequence: descending workload/(length × time window)")
    print("  Berth search: 10 m candidate-position increments")
    print("  Crane rule: maximize average productivity per assigned crane")
    print("  Timeline: prohibit simultaneous occupancy of overlapping berth intervals")
    print("  Runtime expression: O(n log n + n²P + nK), for n vessels, P positions, K crane counts")

    print("\nSolution metrics:")
    print(f"  Total cost: ${metrics['total_cost']:,.2f}")
    print(f"  Average turnaround: {metrics['avg_turnaround_time']:.2f} hours")
    print(
        "  Aggregate berth-length ratio: "
        f"{metrics['aggregate_berth_length_ratio'] * 100:.2f}%"
    )
    print(f"  Sum of vessel crane assignments: {metrics['total_cranes_used']}")
    print(f"  Measured runtime: {computation_time:.6f} seconds")

    print("\nAssignments:")
    for assignment in assignments:
        vessel = assignment.vessel
        print(
            f"  Vessel {vessel.id}: berth [{assignment.position:.0f}, "
            f"{assignment.position + vessel.length:.0f}] m; "
            f"start {assignment.start_time:.2f} h; "
            f"finish {assignment.finish_time:.2f} h; "
            f"{assignment.num_cranes} cranes; "
            f"priority {vessel.priority:.4f}"
        )

    assert len(assignments) == len(instance.vessels)
    print("\nInterpretation:")
    print("  The schedule is feasible under the stated berth and service assumptions.")
    print("  The constructive heuristic does not guarantee a globally optimal solution.")
    print("  The planning horizon is descriptive and is not enforced as a deadline.")


final_heuristic_summary(instance, assignments, metrics, comp_time)